In [0]:
%sql
CREATE OR REPLACE TEMP VIEW bm_rows AS
            SELECT 'motor' AS line, row_id, policy_id, period_start, period_end, exposure FROM workspace.iaa.policies_motor_clean
  UNION ALL SELECT 'buildings', row_id, policy_id, period_start, period_end, exposure FROM workspace.iaa.policies_buildings_clean
  UNION ALL SELECT 'contents',  row_id, policy_id, period_start, period_end, exposure FROM workspace.iaa.policies_contents_clean;

CREATE OR REPLACE TEMP VIEW bm_year AS
WITH cost AS (
  SELECT r.line, year(r.period_start) AS yr, SUM(c.incurred_to_date - c.recoveries) AS cost
  FROM bm_rows r
  JOIN workspace.iaa.claims_clean c
    ON c.line = r.line AND c.policy_id = r.policy_id AND c.loss_date BETWEEN r.period_start AND r.period_end
  WHERE c.status <> 'repudiated'
  GROUP BY 1, 2),
expo AS (SELECT line, year(period_start) AS yr, SUM(exposure) AS exposure FROM bm_rows GROUP BY 1, 2)
SELECT e.line, e.yr, e.exposure, c.cost, c.cost / e.exposure AS cost_per_policy_year
FROM expo e JOIN cost c USING (line, yr);

-- Compare with the values computed independently from the raw files
CREATE OR REPLACE TEMP VIEW bm_year_check AS
WITH expected (line, yr, expected_bc) AS (
  VALUES ('buildings', 2021, 383.94), ('buildings', 2022, 403.12), ('buildings', 2023, 432.25), ('buildings', 2024, 449.27),
         ('contents',  2021, 105.45), ('contents',  2022, 105.48), ('contents',  2023, 116.95), ('contents',  2024, 116.80),
         ('motor',     2021, 563.31), ('motor',     2022, 575.32), ('motor',     2023, 587.47), ('motor',     2024, 453.82)
)
SELECT e.line, e.yr, ROUND(y.exposure) AS exposure, ROUND(y.cost) AS cost,
       e.expected_bc, ROUND(y.cost_per_policy_year, 2) AS actual_bc,
       CASE WHEN ABS(y.cost_per_policy_year - e.expected_bc) <= 0.01 THEN 'PASS' ELSE 'FAIL' END AS status
FROM expected e LEFT JOIN bm_year y USING (line, yr);

SELECT * FROM bm_year_check ORDER BY line, yr;

-- Stop the notebook if anything differs
SELECT CASE WHEN COUNT_IF(status <> 'PASS') > 0
            THEN raise_error('Benchmark inputs differ from expected: see the table above')
            ELSE 'All 12 line-years match' END AS result
FROM bm_year_check;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW bm_2025 AS
WITH base AS (
  SELECT line,
         MAX(CASE WHEN yr = 2021 THEN cost_per_policy_year END) AS bc21,
         MAX(CASE WHEN yr = 2022 THEN cost_per_policy_year END) AS bc22,
         MAX(CASE WHEN yr = 2023 THEN cost_per_policy_year END) AS bc23
  FROM bm_year GROUP BY line),
trend AS (SELECT *, pow(bc23 / bc21, 0.5) - 1 AS g FROM base),
test AS (
            SELECT 'motor' AS line, SUM(exposure) AS exposure_2025, COUNT(*) AS test_rows FROM workspace.iaa.test_motor_clean
  UNION ALL SELECT 'buildings', SUM(exposure), COUNT(*) FROM workspace.iaa.test_buildings_clean
  UNION ALL SELECT 'contents',  SUM(exposure), COUNT(*) FROM workspace.iaa.test_contents_clean),
scen AS (
  SELECT t.*, s.scenario,
         CASE s.scenario WHEN '1. low' THEN 0 WHEN '2. central' THEN t.g ELSE t.g + 0.02 END AS tr
  FROM trend t CROSS JOIN (VALUES ('1. low'), ('2. central'), ('3. high')) AS s(scenario)),
bm AS (
  SELECT line, scenario, tr,
         (bc21 * pow(1 + tr, 4) + bc22 * pow(1 + tr, 3) + bc23 * pow(1 + tr, 2)) / 3 AS bc_2025
  FROM scen)
SELECT b.line, b.scenario, b.tr AS annual_trend, b.bc_2025,
       x.exposure_2025 * b.bc_2025 / 1e6         AS total_2025_millions,
       x.exposure_2025 * b.bc_2025 / x.test_rows AS mean_per_test_row
FROM bm b JOIN test x USING (line);

-- Compare with the values computed independently from the raw files
CREATE OR REPLACE TEMP VIEW bm_2025_check AS
WITH expected (line, scenario, exp_bc, exp_total_m, exp_mean) AS (
  VALUES ('buildings', '1. low',     406.4, 61.07, 331.6),
         ('buildings', '2. central', 484.9, 72.86, 395.6),
         ('buildings', '3. high',    512.9, 77.07, 418.5),
         ('contents',  '1. low',     109.3, 16.41,  87.5),
         ('contents',  '2. central', 127.5, 19.15, 102.1),
         ('contents',  '3. high',    135.0, 20.26, 108.1),
         ('motor',     '1. low',     575.4, 115.06, 451.3),
         ('motor',     '2. central', 612.7, 122.53, 480.5),
         ('motor',     '3. high',    649.5, 129.89, 509.4)
)
SELECT e.line, e.scenario, ROUND(b.annual_trend, 4) AS annual_trend,
       e.exp_bc,      ROUND(b.bc_2025, 1)             AS cost_per_policy_year_2025,
       e.exp_total_m, ROUND(b.total_2025_millions, 2) AS total_2025_millions,
       e.exp_mean,    ROUND(b.mean_per_test_row, 1)   AS mean_per_test_row,
       CASE WHEN ABS(b.bc_2025 - e.exp_bc) <= 0.1
             AND ABS(b.total_2025_millions - e.exp_total_m) <= 0.01
             AND ABS(b.mean_per_test_row - e.exp_mean) <= 0.1
            THEN 'PASS' ELSE 'FAIL' END AS status
FROM expected e LEFT JOIN bm_2025 b USING (line, scenario);

SELECT * FROM bm_2025_check ORDER BY line, scenario;

-- Stop the notebook if anything differs
SELECT CASE WHEN COUNT_IF(status <> 'PASS') > 0
            THEN raise_error('Benchmark differs from expected: see the table above')
            ELSE 'All 9 line-scenarios match' END AS result
FROM bm_2025_check;

In [0]:
%sql
-- The only line to edit: point it at the uploaded submission
CREATE OR REPLACE TEMP VIEW sub AS SELECT * FROM workspace.iaa.sample_submission; -- update with submission to check. 

-- A. Is the file valid for Kaggle?
CREATE OR REPLACE TEMP VIEW sub_valid AS
WITH test_ids AS (
            SELECT row_id FROM workspace.iaa.test_motor_clean
  UNION ALL SELECT row_id FROM workspace.iaa.test_buildings_clean
  UNION ALL SELECT row_id FROM workspace.iaa.test_contents_clean),
c AS (
  SELECT (SELECT COUNT(*) FROM sub)                                              AS rows_in_file,
         (SELECT COUNT(DISTINCT row_id) FROM sub)                                AS distinct_row_ids,
         (SELECT COUNT(*) FROM test_ids t LEFT ANTI JOIN sub s USING (row_id))   AS test_rows_missing,
         (SELECT COUNT(*) FROM sub s LEFT ANTI JOIN test_ids t USING (row_id))   AS unknown_row_ids,
         (SELECT COUNT_IF(prediction IS NULL OR prediction < 0) FROM sub)        AS null_or_negative,
         (SELECT COUNT_IF(prediction = 0) FROM sub)                              AS exact_zeros)
SELECT *, CASE WHEN rows_in_file = 626639 AND distinct_row_ids = 626639 AND test_rows_missing = 0
                AND unknown_row_ids = 0 AND null_or_negative = 0 AND exact_zeros = 0
               THEN 'PASS' ELSE 'FAIL' END AS status
FROM c;

SELECT * FROM sub_valid;

-- B. Is the level plausible? Each line's total against the benchmark range
WITH s AS (
  SELECT CASE left(row_id, 1) WHEN 'M' THEN 'motor' WHEN 'B' THEN 'buildings' ELSE 'contents' END AS line,
         COUNT(*) AS rows, SUM(prediction) / 1e6 AS total_m, AVG(prediction) AS mean_per_row
  FROM sub GROUP BY 1),
b AS (
  SELECT line,
         MAX(CASE WHEN scenario = '1. low'     THEN total_2025_millions END) AS low_m,
         MAX(CASE WHEN scenario = '2. central' THEN total_2025_millions END) AS central_m,
         MAX(CASE WHEN scenario = '3. high'    THEN total_2025_millions END) AS high_m
  FROM bm_2025 GROUP BY line)
SELECT s.line, s.rows, ROUND(s.mean_per_row, 1) AS mean_per_row,
       ROUND(s.total_m, 2) AS submission_total_m,
       ROUND(b.low_m, 2) AS low_m, ROUND(b.central_m, 2) AS central_m, ROUND(b.high_m, 2) AS high_m,
       ROUND(s.total_m / b.central_m - 1, 3) AS vs_central,
       CASE WHEN s.total_m < b.low_m  THEN 'BELOW range'
            WHEN s.total_m > b.high_m THEN 'ABOVE range'
            ELSE 'in range' END AS level_check
FROM s JOIN b USING (line)
ORDER BY s.line;